# Trucki intelligence: reproducible audit checks

## Summary
These checks exercise existing calculation code using invented records. They test software
behaviour and analytical limitations, not customer performance, statutory law or demand.
Exact results follow below. No database queries or network requests are made.

## Context and methods
Scope: local working tree on 2026-10-07, including the uncommitted consignment foundation.
Sources: `backend/regulatory/modelling.py`, `backend/yard/reports.py` and their evaluator imports.
Run `backend/.venv/Scripts/python.exe docs/analysis/run_intelligence_audit.py` from the repo root.
The script executes every code cell in order and saves outputs plus a source fingerprint.
Jupyter packages are absent, so execution uses Python compile/exec, not a Jupyter kernel.
The notebook uses only plain text tables; no charts are needed for these three bounded checks.

### Key assumptions
The capacity comparison uses 20 paired seeds, 72 invented arrivals, 36 arrivals/hour and
20-minute mean service. Only synthetic PASS cases consume docks. These are stress-test inputs,
not estimates of any tenant. Invalid timestamps and legacy milestones below are invented.


In [1]:
from pathlib import Path
import os, sys, json, statistics, hashlib
from datetime import datetime, timedelta, timezone
from types import SimpleNamespace
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'backend/regulatory/modelling.py').exists())
sys.path.insert(0, str(root / 'backend'))
os.environ['DJANGO_SETTINGS_MODULE'] = 'spotter_backend.settings'
os.environ['DJANGO_SECRET_KEY'] = 'synthetic-intelligence-audit-only'
os.environ['DATABASE_URL'] = 'sqlite:///:memory:'
import django
django.setup()
from regulatory.modelling import run_model
from yard.reports import compute_turnaround_stats
results = {'synthetic': True, 'customer_data': False, 'verified_law': False}
sources = ['backend/regulatory/modelling.py', 'backend/yard/reports.py',
           'backend/regulatory/services.py', 'backend/regulatory/engine/evaluator.py',
           'backend/compliance/policy.py']
results['source_sha256'] = {p: hashlib.sha256((root / p).read_bytes()).hexdigest() for p in sources}
print('Local source fingerprints captured. Synthetic inputs only.')


Local source fingerprints captured. Synthetic inputs only.


## Results
### 1. Capacity sensitivity, not a forecast
Compare identical random seeds and arrival/decision/service draws across dock counts.
The min/max columns describe seed variation, not confidence intervals for real operations.


In [2]:
runs = {d: [run_model(seed=s, vehicles=72, docks=d, arrivals_per_hour=36, service_minutes=20)
            for s in range(20)] for d in (2, 3, 4)}
summary = []
for d, values in runs.items():
    waits = [r['summary']['average_wait_minutes'] for r in values]
    summary.append({'docks': d, 'mean_of_run_mean_wait_minutes': round(statistics.mean(waits), 2),
                    'min_run_mean_wait_minutes': min(waits), 'max_run_mean_wait_minutes': max(waits),
                    'mean_blocked': statistics.mean(r['summary']['blocked'] for r in values)})
for seed in range(20):
    baseline = runs[2][seed]
    for d in (3, 4):
        candidate = runs[d][seed]
        assert candidate['summary']['decisions'] == baseline['summary']['decisions']
        assert candidate['summary']['average_wait_minutes'] <= baseline['summary']['average_wait_minutes']
        assert candidate['summary']['scenario_checks_passed'] == 7
assert run_model(seed=42)['result_digest'] == run_model(seed=42)['result_digest']
results['capacity_stress_test'] = summary
results['capacity_checks'] = {'paired_seeds': 20, 'runs': 60, 'seven_cases_match_all_runs': True,
                             'blocked_counts_unchanged_by_dock_count': True, 'repeatable_digest': True}
print(json.dumps(summary, indent=2))
print('All 60 runs match seven invented evaluation cases. Added docks do not resolve blocked cases.')


[
  {
    "docks": 2,
    "mean_of_run_mean_wait_minutes": 151.08,
    "min_run_mean_wait_minutes": 114.5,
    "max_run_mean_wait_minutes": 192.1,
    "mean_blocked": 28.75
  },
  {
    "docks": 3,
    "mean_of_run_mean_wait_minutes": 80.42,
    "min_run_mean_wait_minutes": 55.17,
    "max_run_mean_wait_minutes": 111.38,
    "mean_blocked": 28.75
  },
  {
    "docks": 4,
    "mean_of_run_mean_wait_minutes": 46.33,
    "min_run_mean_wait_minutes": 25.37,
    "max_run_mean_wait_minutes": 71.16,
    "mean_blocked": 28.75
  }
]
All 60 runs match seven invented evaluation cases. Added docks do not resolve blocked cases.


### 2. Hidden denominator and milestone differences
Test one physical exit, one legacy completion proxy, one active movement, one reversed timestamp
and one missing timestamp. Report total counts all five, while the duration means exclude two.
A completion proxy is a different event from physical exit and must be segmented in benchmarks.


In [3]:
now = datetime(2026, 10, 7, 12, tzinfo=timezone.utc)
def row(status, entered, exited=None, updated=None, semantics='SEPARATE_V1'):
    return SimpleNamespace(status=status, entry_timestamp=entered, exit_timestamp=exited,
                           updated_at=updated, milestone_semantics=semantics)
sample = [row('RELEASED', now-timedelta(minutes=120), now-timedelta(minutes=20)),
          row('COMPLETED', now-timedelta(minutes=120), updated=now-timedelta(minutes=100), semantics='LEGACY_COMBINED'),
          row('QUEUED', now-timedelta(minutes=75)),
          row('RELEASED', now-timedelta(minutes=10), now-timedelta(minutes=20)),
          row('QUEUED', None)]
mixed = compute_turnaround_stats(sample, now)
physical = compute_turnaround_stats(sample[:1], now)
assert mixed['total'] == 5 and mixed['avgTurnaroundMinutes'] == 60
assert physical['avgTurnaroundMinutes'] == 100
results['duration_coverage'] = {'reported': mixed, 'physical_exit_only': physical,
                               'valid_duration_rows': 3, 'excluded_duration_rows': 2,
                               'legacy_proxy_completed_rows': 1}
print(json.dumps(results['duration_coverage'], indent=2))


{
  "reported": {
    "total": 5,
    "byStatus": {
      "RELEASED": 2,
      "COMPLETED": 1,
      "QUEUED": 2
    },
    "avgWaitMinutes": 75.0,
    "avgTurnaroundMinutes": 60.0,
    "overdueCount": 1
  },
  "physical_exit_only": {
    "total": 1,
    "byStatus": {
      "RELEASED": 1
    },
    "avgWaitMinutes": null,
    "avgTurnaroundMinutes": 100.0,
    "overdueCount": 0
  },
  "valid_duration_rows": 3,
  "excluded_duration_rows": 2,
  "legacy_proxy_completed_rows": 1
}


### 3. Tenant threshold changes the action count
A 75-minute active movement breaches 60 minutes but not 90 minutes. This is a policy choice,
not a statistical risk estimate; thresholds need tenant/site/service definitions and versions.


In [4]:
active = [row('QUEUED', now-timedelta(minutes=75))]
counts = {str(limit): compute_turnaround_stats(active, now, overdue_minutes=limit)['overdueCount']
          for limit in (60, 90)}
assert counts == {'60': 1, '90': 0}
results['threshold_sensitivity'] = counts
print(json.dumps(counts, indent=2))
(root / 'docs/analysis/intelligence-audit-results.json').write_text(json.dumps(results, indent=2) + '\n', encoding='utf-8')


{
  "60": 1,
  "90": 0
}


## Takeaways
The model supports reproducible what-if comparisons within its assumptions. Dock capacity does
not fix evidence/policy holds. The report helper mixes legacy completion proxies with actual exits
and lacks duration-coverage denominators; segment them before performance benchmarking.
Tenant thresholds change which movements demand attention. None of these checks proves an
unmet customer need, production adapter reliability, prediction accuracy or financial benefit.
